In [75]:
import pandas as pd
import numpy as np

In [76]:
def to_mfp_html(filename,title,df):
    html_string = '''
    <html>
      <head><title>{title}</title></head>
      <link rel="stylesheet" type="text/css" href="mfp_style.css"/>
      <body>
        {table}
      </body>
    </html>.
    '''

    with open(filename+'.html', 'w') as f:
        f.write(html_string.format(title=title,
                                   table=df.to_html(classes='mfp_style',render_links=True,escape=False,sparsify=True,index=False)))

In [77]:
import pickle

with open('season_id_dict.pickle', 'rb') as f:
    season_ids = pickle.load(f)

In [78]:
mfp_df = pd.read_csv('mfp_all_seasons.csv')
mfp_df.shape

(1190, 41)

In [79]:
mfp_df['series_id'] = mfp_df.season_id.map(str)

In [80]:
mfp_full = pd.read_csv('mfp_all_seasons_full.csv')
mfp_full['7pts'] = mfp_full.points == 7
mfp_full['5pts'] = mfp_full.points == 5
mfp_full['4pts'] = mfp_full.points == 4
mfp_full['3pts'] = mfp_full.points == 3
mfp_full['1pts'] = mfp_full.points == 1

In [81]:
mfp_df = pd.merge(mfp_df,mfp_full.groupby(['players','season_id']).Arena.count().reset_index().rename(columns=dict(players='name',Arena='rounds_played')),
         on=['name','season_id'])

In [82]:
mfp_full = mfp_full.groupby(['players','season_id']).agg({
    'Time': 'sum',
    'adj_time': 'mean',
    'group_number':'mean',
    'num_players':'mean',
    'points': 'mean',
    'positions':'mean',
    '7pts':'sum',
    '5pts':'sum',
    '4pts':'sum',
    '3pts':'sum',
    '1pts':'sum'
}).reset_index().rename(columns=dict(
players='name',Time='total_time_played_secs',adj_time='time_per_game_secs',group_number='avg_group_num',
    num_players='avg_num_players_in_game',points='avg_pts_per_game',positions='avg_position_value'))
mfp_full.shape

(1187, 13)

In [83]:
mfp_df = pd.merge(mfp_df,mfp_full,on=['name','season_id'])

In [84]:
all_weeks = ['week1','week2','week3','week4','week5','week6','week7','week8','week9','week10']
all_pts = ['7pts','5pts','4pts','3pts','1pts']

In [85]:
mfp_df['season'] = mfp_df.series_id.map(season_ids)

In [86]:
mfp_df = mfp_df[['name','season']+all_weeks+['avg_att', 'avg_score', 'cur_result', 'cur_total', 'current_score',
                                    'kept_score', 'kept_score_att', 'max_score_possible', 'mean_adj_score',
                                    'mean_adj_score_att','proj_att', 'proj_score','proj_score_att', 'proj_type',
                                    'remaining_weeks_above_avg','required_avg_trophy', 'required_avg_win', 'score_rank',
                                    'score_to_beat', 'season_id', 'top_score_1', 'top_score_2','top_score_3',
                                    'top_score_4', 'top_score_5', 'top_score_6', 'weeks_left_att', 'weeks_played',
                                    'weeks_won', 'series_id','rounds_played', 'total_time_played_secs', 'time_per_game_secs',
                                    'avg_group_num', 'avg_num_players_in_game', 'avg_pts_per_game', 'avg_position_value',
                                    '7pts', '5pts', '4pts', '3pts', '1pts']].rename(columns=dict(score_rank='season_ranking'))

In [87]:
temp = mfp_df.groupby('season').agg({
    'name':'count',
    'season_id': 'min'}).reset_index().rename(columns=dict(name='# Players'))
temp.sort_values('season_id',inplace=True)
temp2 = mfp_df[mfp_df.weeks_played >= 0].groupby('season').agg({
    'name':'count'
}).rename(columns=dict(name='# Players Submitted to IFPA'))


In [88]:
temp

,season,# Players,season_id
14,Winter 17,27,209
5,Spring 17,31,285
10,Summer 17,41,350
0,Fall 17,38,409
15,Winter 18,31,475
6,Spring 18,53,582
11,Summer 18,59,672
1,Fall 18,53,748
16,Winter 19,60,861
7,Spring 19,71,987


In [89]:
temp = pd.merge(temp,temp2,left_on='season',right_index=True)

In [90]:
def make_season_index_urls(row):
    row['Matchplay Link'] = f'<a href=https://matchplay.events/live/series/'+str(row.season_id)+'>'+row.season+' on Matchplay</a>'
    row['MF-Stats Link'] = f'<a href=/stats/mfp_league_'+str(row.season_id)+'_standings.html>'+row.season+'</a>'
    return row
                                                      
to_mfp_html('seasons','MFP Season Index',temp.apply(make_season_index_urls,axis=1)[['MF-Stats Link','# Players','# Players Submitted to IFPA','Matchplay Link']])

In [91]:
temp

,season,# Players,season_id,# Players Submitted to IFPA
14,Winter 17,27,209,27
5,Spring 17,31,285,31
10,Summer 17,41,350,41
0,Fall 17,38,409,38
15,Winter 18,31,475,31
6,Spring 18,53,582,53
11,Summer 18,59,672,59
1,Fall 18,53,748,53
16,Winter 19,60,861,60
7,Spring 19,71,987,71


In [92]:
lowest_scores = mfp_df[(mfp_df.current_score <= 8)&(mfp_df.rounds_played >= 5)][['name','season','current_score','rounds_played']+all_weeks]
to_mfp_html('season_scores_less_than_8',"Lowest Season Scores - Min 5 Rounds Played",lowest_scores)

In [93]:
mfp_df['first_half'] = mfp_df.loc[:,['week1','week2','week3','week4','week5']].sum(axis=1)
mfp_df['second_half'] = mfp_df.loc[:,['week6','week7','week8','week9','week10']].sum(axis=1)

In [94]:
mfp_df.sort_values('first_half',ascending=False)[['name','season','first_half','second_half','current_score']+all_weeks].iloc[:10]

,name,season,first_half,second_half,current_score,week1,week2,week3,week4,week5,week6,week7,week8,week9,week10
137,Philip Ryder,Winter 18,147.0,79.0,153.0,33.0,29.0,25.0,27.0,33.0,31.0,NaN,NaN,23.0,25.0
995,Matthew Talley,Fall 22,143.0,114.0,170.0,33.0,27.0,27.0,25.0,31.0,25.0,27.0,19.0,23.0,20.0
825,Cary Carmichael,Spring 22,141.0,113.0,172.0,33.0,23.0,35.0,27.0,23.0,29.0,21.0,19.0,25.0,19.0
466,Lucy Cachux,Summer 19,139.0,102.0,139.0,27.0,33.0,29.0,27.0,23.0,21.0,21.0,21.0,19.0,20.0
911,Greg Dunlap,Summer 22,137.0,45.0,161.0,27.0,27.0,31.0,31.0,21.0,23.0,NaN,22.0,NaN,NaN
221,Tony Lavigna,Summer 18,137.0,125.0,145.0,31.0,25.0,29.0,27.0,25.0,23.0,19.0,25.0,33.0,25.0
824,John Tracey,Spring 22,137.0,95.0,175.0,25.0,31.0,21.0,33.0,27.0,19.0,NaN,29.0,17.0,30.0
280,Tony Lavigna,Fall 18,135.0,131.0,149.0,21.0,35.0,25.0,25.0,29.0,29.0,23.0,29.0,27.0,23.0
910,Matthew Talley,Summer 22,132.0,77.0,165.0,27.0,26.0,25.0,27.0,27.0,19.0,31.0,27.0,NaN,NaN
750,Xavier Marin,Winter 22,131.0,149.0,184.0,27.0,29.0,27.0,29.0,19.0,33.0,33.0,33.0,25.0,25.0


In [95]:
mfp_df.sort_values('second_half',ascending=False)[['name','season','first_half','second_half','current_score']+all_weeks].iloc[:10]

,name,season,first_half,second_half,current_score,week1,week2,week3,week4,week5,week6,week7,week8,week9,week10
750,Xavier Marin,Winter 22,131.0,149.0,184.0,27.0,29.0,27.0,29.0,19.0,33.0,33.0,33.0,25.0,25.0
169,Tony Lavigna,Spring 18,61.0,139.0,145.0,25.0,19.0,NaN,17.0,NaN,25.0,29.0,33.0,33.0,19.0
58,Tony Lavigna,Summer 17,100.0,137.0,153.0,NaN,15.0,25.0,27.0,33.0,29.0,33.0,23.0,21.0,31.0
99,Fred Hamilton,Fall 17,129.0,135.0,140.0,27.0,24.0,25.0,26.0,27.0,25.0,25.0,25.0,31.0,29.0
467,Matthew Talley,Summer 19,119.0,135.0,139.0,25.0,27.0,23.0,21.0,23.0,29.0,27.0,31.0,23.0,25.0
465,Will Chernetsky,Summer 19,108.0,135.0,140.0,23.0,25.0,22.0,19.0,19.0,26.0,20.0,29.0,35.0,25.0
28,Matthew Talley,Spring 17,82.0,132.0,133.0,25.0,26.0,NaN,23.0,8.0,29.0,27.0,25.0,26.0,25.0
334,Lucy Cachux,Winter 19,130.0,131.0,143.0,25.0,27.0,23.0,29.0,26.0,27.0,29.0,31.0,23.0,21.0
280,Tony Lavigna,Fall 18,135.0,131.0,149.0,21.0,35.0,25.0,25.0,29.0,29.0,23.0,29.0,27.0,23.0
826,Tony Lavigna,Spring 22,125.0,131.0,164.0,23.0,25.0,23.0,23.0,31.0,25.0,23.0,31.0,29.0,23.0


In [96]:
def biggest_drop(row):
    temp = row.filter(regex='^week\d+$')
    result = []
    for x in range(len(temp)):
        result.append(temp[x-1]-temp[x])
    row['biggest_drop'] = max(result)
    return row

mfp_df = mfp_df.apply(biggest_drop,axis=1)

In [97]:
to_mfp_html(filename='biggest_week_to_week_drop.html',title='Biggest Week to Week Drop',df=mfp_df.sort_values('biggest_drop',ascending=False)[['name','season','biggest_drop']+all_weeks][:10])

In [98]:
def season_std(row):
    row['std'] = row.filter(regex='^week\d+$').std()
    return row

mfp_df = mfp_df.apply(season_std,axis=1)

In [99]:
mfp_df[(mfp_df['std']> 0)&(mfp_df.weeks_played >= 5)].sort_values('std',ascending=False)[['name','season','std','avg_score']+all_weeks+['season_ranking']][:10]

,name,season,std,avg_score,week1,week2,week3,week4,week5,week6,week7,week8,week9,week10,season_ranking
764,Michael Mattsson,Winter 22,10.590877,22.833333,NaN,27.0,NaN,29.0,23.0,2.0,NaN,25.0,31.0,NaN,15.0
758,Philip Ryder,Winter 22,9.110434,20.333333,27.0,17.0,29.0,27.0,29.0,15.0,17.0,21.0,NaN,1.0,9.0
627,Nick Vibe,Winter 20,8.763561,17.600000,NaN,21.0,23.0,2.0,NaN,NaN,NaN,21.0,21.0,NaN,31.0
281,Nick Fitzpatrick,Fall 18,8.648699,25.000000,35.0,21.0,NaN,NaN,26.0,NaN,NaN,35.0,14.0,19.0,2.0
307,Cody Reeves,Fall 18,8.590693,16.400000,25.0,10.0,NaN,NaN,NaN,5.0,19.0,23.0,NaN,NaN,28.0
569,Barrett Lindstrom,Fall 19,8.532292,16.400000,19.0,NaN,NaN,NaN,17.0,9.0,29.0,NaN,8.0,NaN,33.0
112,Sneha Kolar,Fall 17,8.334667,17.333333,NaN,31.0,NaN,16.0,12.0,NaN,14.0,8.0,NaN,23.0,14.0
250,Ikela Lagapa,Summer 18,7.861298,16.400000,17.0,10.0,NaN,25.0,23.0,NaN,7.0,NaN,NaN,NaN,30.0
62,Michael Mattsson,Summer 17,7.797435,24.400000,17.0,31.0,25.0,33.0,NaN,NaN,NaN,NaN,16.0,NaN,5.0
948,Harry Franklin,Summer 22,7.536577,17.600000,6.0,19.0,17.0,NaN,19.0,NaN,NaN,27.0,NaN,NaN,40.0


In [100]:
def weeks_above_22(row):
    row['weeks_above_22'] = sum(row.filter(regex='^week\d+$') > 22)
    return row

In [101]:
mfp_df = mfp_df.apply(weeks_above_22,axis=1)

In [102]:
mfp_df.sort_values('weeks_above_22',ascending=False)[['name','season','current_score','weeks_above_22']+all_weeks]

,name,season,current_score,weeks_above_22,week1,week2,week3,week4,week5,week6,week7,week8,week9,week10
826,Tony Lavigna,Spring 22,164.0,10,23.0,25.0,23.0,23.0,31.0,25.0,23.0,31.0,29.0,23.0
99,Fred Hamilton,Fall 17,140.0,10,27.0,24.0,25.0,26.0,27.0,25.0,25.0,25.0,31.0,29.0
334,Lucy Cachux,Winter 19,143.0,9,25.0,27.0,23.0,29.0,26.0,27.0,29.0,31.0,23.0,21.0
467,Matthew Talley,Summer 19,139.0,9,25.0,27.0,23.0,21.0,23.0,29.0,27.0,31.0,23.0,25.0
280,Tony Lavigna,Fall 18,149.0,9,21.0,35.0,25.0,25.0,29.0,29.0,23.0,29.0,27.0,23.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
528,Danielle Licata,Summer 19,13.0,0,NaN,NaN,NaN,NaN,NaN,NaN,13.0,NaN,NaN,NaN
529,Paul Tolbert,Summer 19,13.0,0,NaN,NaN,13.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
530,Filimon Garcia,Summer 19,11.0,0,NaN,NaN,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
531,Julie Gomez,Summer 19,11.0,0,NaN,NaN,NaN,NaN,11.0,NaN,NaN,NaN,NaN,NaN


In [103]:
def get_all_best_game_options(row):
    sorted_games = sorted(row.filter(regex='^week\d+$').fillna(0),reverse=True)
    if(len(sorted_games) <5):
        row['best5_score'] = sum(sorted_games)
        row['best6_score'] = sum(sorted_games)
    row['best5_score'] = sum(sorted_games[:5])
    row['best6_score'] = sum(sorted_games[:6])
    
    return row
    
mfp_df = mfp_df.apply(get_all_best_game_options,axis=1)

In [104]:
mfp_df.sort_values('best6_score',ascending=False)[['name','season','season_ranking','cur_total']+all_pts+['std','weeks_above_22','best5_score','best6_score']][:20]

,name,season,season_ranking,cur_total,7pts,5pts,4pts,3pts,1pts,std,weeks_above_22,best5_score,best6_score
750,Xavier Marin,Winter 22,1.0,280.0,28,12,0,7,3,4.447221,9,157.0,184.0
58,Tony Lavigna,Summer 17,1.0,237.0,25,9,0,3,8,6.000000,7,153.0,178.0
137,Philip Ryder,Winter 18,1.0,226.0,23,9,0,6,2,3.845220,8,153.0,178.0
824,John Tracey,Spring 22,1.0,232.0,22,9,3,5,6,5.651942,6,150.0,175.0
280,Tony Lavigna,Fall 18,1.0,266.0,24,15,0,6,5,4.087923,9,149.0,174.0
597,Michael Mattsson,Winter 20,1.0,239.0,22,11,2,6,4,3.876568,7,147.0,172.0
825,Cary Carmichael,Spring 22,2.0,254.0,20,17,0,9,4,5.561774,7,149.0,172.0
996,Philip Priddy,Fall 22,2.0,221.0,17,13,0,11,4,6.463573,6,145.0,170.0
995,Matthew Talley,Fall 22,1.0,257.0,24,12,1,6,7,4.372896,8,145.0,170.0
909,John Tracey,Summer 22,1.0,195.0,17,14,0,1,3,3.023716,7,145.0,170.0


In [105]:
mfp_df.columns

Index(['name', 'season', 'week1', 'week2', 'week3', 'week4', 'week5', 'week6',
       'week7', 'week8', 'week9', 'week10', 'avg_att', 'avg_score',
       'cur_result', 'cur_total', 'current_score', 'kept_score',
       'kept_score_att', 'max_score_possible', 'mean_adj_score',
       'mean_adj_score_att', 'proj_att', 'proj_score', 'proj_score_att',
       'proj_type', 'remaining_weeks_above_avg', 'required_avg_trophy',
       'required_avg_win', 'season_ranking', 'score_to_beat', 'season_id',
       'top_score_1', 'top_score_2', 'top_score_3', 'top_score_4',
       'top_score_5', 'top_score_6', 'weeks_left_att', 'weeks_played',
       'weeks_won', 'series_id', 'rounds_played', 'total_time_played_secs',
       'time_per_game_secs', 'avg_group_num', 'avg_num_players_in_game',
       'avg_pts_per_game', 'avg_position_value', '7pts', '5pts', '4pts',
       '3pts', '1pts', 'first_half', 'second_half', 'biggest_drop', 'std',
       'weeks_above_22', 'best5_score', 'best6_score'],
      dtype=

In [106]:
temp = mfp_df.groupby('name').agg({'cur_total':'sum','season_id':'min','total_time_played_secs':'sum', 'weeks_played':'sum','weeks_won':'sum', 'season_ranking':'mean'}).reset_index().rename(columns=dict(score_rank='median_lg_rank'))
temp.season_id = temp.season_id.map(str).map(season_ids)
temp['Pts Per Week'] = temp.cur_total/temp.weeks_played
temp['Time Played Per Week \(secs\)'] = temp.total_time_played_secs/temp.weeks_played
temp['Weekly Winning %'] = temp.weeks_won*100/temp['weeks_played']
temp.rename(columns=dict(name='Player Name',season_id='First Season',cur_total='Total Points',weeks_played = 'Weeks Played', season_ranking = 'Avg Season Ranking'),inplace=True)
temp.sort_values('Weekly Winning %',ascending=False,inplace=True)
temp[temp['Weeks Played'] > 5][:20]

,Player Name,Total Points,First Season,total_time_played_secs,Weeks Played,weeks_won,Avg Season Ranking,Pts Per Week,Time Played Per Week \(secs\),Weekly Winning %
354,Tony Lavigna,2915.0,Winter 17,594829,118,25,9.588235,24.703390,5040.923729,21.186441
250,Michael Mattsson,1433.0,Summer 17,306709,57,12,12.500000,25.140351,5380.859649,21.052632
86,Dale Green,144.0,Winter 20,41313,6,1,55.500000,24.000000,6885.500000,16.666667
264,Nick Fitzpatrick,906.0,Summer 18,175943,38,6,20.571429,23.842105,4630.078947,15.789474
132,Greg Dunlap,1276.0,Fall 19,267876,52,8,17.222222,24.538462,5151.461538,15.384615
71,Cary Carmichael,3716.0,Winter 17,790768,160,23,7.950000,23.225000,4942.300000,14.375000
283,Philip Ryder,2908.0,Winter 17,595987,126,17,14.888889,23.079365,4730.055556,13.492063
177,John Tracey,4131.0,Winter 17,869462,175,21,4.500000,23.605714,4968.354286,12.000000
372,Xavier Marin,1247.0,Summer 19,283488,51,6,13.875000,24.450980,5558.588235,11.764706
244,Matthew Talley,4224.0,Winter 17,896362,177,20,5.700000,23.864407,5064.192090,11.299435


In [107]:
temp = mfp_df.groupby('name').agg({'cur_total':'sum','season_id':'min','total_time_played_secs':'sum', 'weeks_played':'sum','weeks_won':'sum', 'season_ranking':'mean'}).sort_values('cur_total',ascending=False).reset_index().rename(columns=dict(score_rank='median_lg_rank'))[:20]
temp.season_id = temp.season_id.map(str).map(season_ids)
temp['Pts Per Week'] = temp.cur_total/temp.weeks_played
temp['Time Played Per Week \(secs\)'] = temp.total_time_played_secs/temp.weeks_played
temp.rename(columns=dict(name='Player Name',season_id='First Season',cur_total='Total Points',weeks_played = 'Weeks Played', season_ranking = 'Avg Season Ranking'))

,Player Name,Total Points,First Season,total_time_played_secs,Weeks Played,weeks_won,Avg Season Ranking,Pts Per Week,Time Played Per Week \(secs\)
0,Matthew Talley,4224.0,Winter 17,896362,177,20,5.700000,23.864407,5064.192090
1,John Tracey,4131.0,Winter 17,869462,175,21,4.500000,23.605714,4968.354286
2,Philip Priddy,3765.0,Winter 17,853804,171,16,9.100000,22.017544,4993.005848
3,Cary Carmichael,3716.0,Winter 17,790768,160,23,7.950000,23.225000,4942.300000
4,Nikki Carmichael,3079.0,Winter 17,685018,145,6,12.050000,21.234483,4724.262069
5,Tony Lavigna,2915.0,Winter 17,594829,118,25,9.588235,24.703390,5040.923729
6,Philip Ryder,2908.0,Winter 17,595987,126,17,14.888889,23.079365,4730.055556
7,Will Chernetsky,2855.0,Spring 18,567141,127,10,8.733333,22.480315,4465.677165
8,Lucy Cachux,2800.0,Spring 18,571624,124,11,12.400000,22.580645,4609.870968
9,Fred Hamilton,2578.0,Winter 17,514901,115,12,15.625000,22.417391,4477.400000


In [108]:
mfp_df['perfect_nights'] = (mfp_df.filter(regex='^week\d+$') == 35).apply(sum,axis=1)
mfp_df['33s'] = (mfp_df.filter(regex='^week\d+$') == 33).apply(sum,axis=1)

In [109]:
#perfect nights
mfp_df[mfp_df.perfect_nights > 0].sort_values(['perfect_nights'],ascending=False)[:30][['name','perfect_nights','33s','season']+all_weeks]
to_mfp_html('perfect_nights','Perfect League Nights',mfp_df[mfp_df.perfect_nights > 0].sort_values(['perfect_nights'],ascending=False)[:30][['name','perfect_nights','33s','season']+all_weeks])

In [110]:
## 33s in a season
mfp_df.sort_values(['33s'],ascending=False)[:30][['name','perfect_nights','33s','season']+all_weeks].sort_values(['33s'],ascending=False)[:30]

,name,perfect_nights,33s,season,week1,week2,week3,week4,week5,week6,week7,week8,week9,week10
750,Xavier Marin,0,3,Winter 22,27.0,29.0,27.0,29.0,19.0,33.0,33.0,33.0,25.0,25.0
137,Philip Ryder,0,2,Winter 18,33.0,29.0,25.0,27.0,33.0,31.0,NaN,NaN,23.0,25.0
58,Tony Lavigna,0,2,Summer 17,NaN,15.0,25.0,27.0,33.0,29.0,33.0,23.0,21.0,31.0
169,Tony Lavigna,0,2,Spring 18,25.0,19.0,NaN,17.0,NaN,25.0,29.0,33.0,33.0,19.0
222,Nick Fitzpatrick,0,1,Summer 18,13.0,31.0,21.0,33.0,31.0,25.0,25.0,21.0,22.0,23.0
825,Cary Carmichael,1,1,Spring 22,33.0,23.0,35.0,27.0,23.0,29.0,21.0,19.0,25.0,19.0
686,Matthew Talley,0,1,Fall 21,21.0,33.0,25.0,NaN,25.0,NaN,21.0,27.0,25.0,23.0
824,John Tracey,0,1,Spring 22,25.0,31.0,21.0,33.0,27.0,19.0,NaN,29.0,17.0,30.0
168,Cary Carmichael,0,1,Spring 18,21.0,14.0,27.0,33.0,29.0,27.0,21.0,29.0,25.0,21.0
612,Vincent Ruggiero,0,1,Winter 20,10.0,13.0,17.0,25.0,33.0,17.0,19.0,11.0,19.0,NaN


In [111]:
# 33s all time
mfp_df[['name','perfect_nights','33s','season']+all_weeks].groupby('name').sum().sort_values(['33s'],ascending=False)[:30]

,perfect_nights,33s,week1,week2,week3,week4,week5,week6,week7,week8,week9,week10
name,,,,,,,,,,,,
Tony Lavigna,1,7,215.0,272.0,219.0,296.0,316.0,319.0,272.0,302.0,378.0,326.0
Michael Mattsson,0,4,217.0,232.0,206.0,143.0,95.0,87.0,86.0,125.0,176.0,66.0
Cary Carmichael,1,4,431.0,396.0,386.0,385.0,355.0,387.0,324.0,393.0,303.0,356.0
Xavier Marin,1,3,155.0,164.0,148.0,111.0,117.0,108.0,131.0,123.0,90.0,100.0
Matthew Talley,0,3,466.0,491.0,435.0,374.0,423.0,401.0,448.0,448.0,354.0,384.0
John Tracey,0,2,449.0,476.0,447.0,406.0,414.0,465.0,390.0,371.0,325.0,388.0
Philip Ryder,0,2,347.0,287.0,318.0,287.0,317.0,322.0,272.0,279.0,234.0,245.0
Nick Vibe,0,1,17.0,64.0,67.0,48.0,35.0,56.0,48.0,38.0,64.0,50.0
Greg Friedman,0,1,189.0,118.0,84.0,180.0,159.0,58.0,128.0,118.0,88.0,109.0
